## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [7]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/local-datasets/customer/ads-monetisation/client-suggestions'
print(local_datasource)

/Users/E2074/analytics/customer/Ads-Monetisation/client-suggestions
/Users/E2074/local-datasets/customer/ads-monetisation/client-suggestions


## Datasets & Parameter

In [24]:
## Parameter 
current_date = datetime.now()
start_date = '20240617'
end_date = '20240818'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240617  to  20240818


### Weekly ads data

In [25]:
data_query = f"""

WITH ads AS (

       SELECT 
            DISTINCT
            yyyymmdd,
            DATE_FORMAT(DATE_TRUNC('week', DATE_PARSE(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_date,
            city,
            service,
            userId,
            pagename,
            CASE 
            WHEN pagename IN ('HomeScreen', 'CaptainSearchScreen') THEN pagename
            WHEN lower(slotname) LIKE '%ontheway%' THEN 'onTheWay'
            WHEN lower(slotname) LIKE '%arrived%' THEN 'arrived'
            WHEN lower(slotname) LIKE '%started%' THEN 'started'
            
            WHEN ltrim(orderstatus) IS NOT NULL THEN orderstatus
            END screen_name,
            
            CASE 
            WHEN lower(slotname) LIKE '%-%' THEN substring(slotname, 1, strpos(slotname, ':') - 1)
            ELSE slotname END slotname,
            eventName,
            CASE 
            WHEN hh < '08' THEN 'd. rest'
            WHEN hh >= '08' AND hh <= '11' THEN 'a. morning'
            WHEN hh >= '12' AND hh <= '16' THEN 'b. afternoon'
            WHEN hh >= '17' AND hh <= '22' THEN 'c. evening'
            ELSE 'd. rest' END AS time_temporal,
            adid,
            responseType,
            format
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable_full
        
        WHERE  
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and responseType = 'GAMBanner'
            and city in ('Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Kolkata', 'Jaipur', 'Mumbai')
            and format IN  ('nativeVideoBanner', 'nativeBanner')
            and eventName in ('Ad_Click', 'Ad_Viewable_Impression')
    ),
    
    segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '{segment_date}'
    )
    
    
    SELECT
        yyyymmdd,
        week_date,
        city,
        -- service,
        -- responseType,
        -- format,
        pagename,
        screen_name,
        slotname,
        time_temporal,
        coalesce( ltr_segment, 'UNKNOWN')ltr_segment,
        coalesce( retention_segment, 'UNKNOWN') retention_segment,
        -- coalesce( lifetime_stage, 'UNKNOWN') lifetime_stage,
        count(DISTINCT CASE WHEN eventName = 'Ad_Viewable_Impression' THEN userId END) view,
        count(DISTINCT CASE WHEN eventName = 'Ad_Click' THEN userId END) click
    FROM 
        ads
    LEFT JOIN 
        segment s
        ON userId = s.customer_id
    
    GROUP BY 1,2,3,4,5,6,7,8,9
        
"""

# Execute the query and get the result as a DataFrame
df_data = pd.read_sql(data_query, connection)

df_data.to_csv( local_datasource + '/customer_data_{}_{}.csv'.format(start_date, end_date), index=False)

In [26]:
# df_data = pd.read_csv( local_datasource + '/analysis_data_{}_{}.csv'.format(start_date, end_date))

## User defined function

In [40]:
agg_dict = {
    'view': 'sum',
    'click': 'sum',
}

def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

def calculate_percentage_columns(df):
    df['ctr'] = calculate_percentage(df, 'click', 'view')    
    return df

## Data

In [28]:
df_data.head(5)

,yyyymmdd,week_date,city,pagename,screen_name,slotname,time_temporal,ltr_segment,retention_segment,view,click
0,20240620,20240617,Delhi,PostOrderScreen,onTheWay,PostOrderOnTheWay2,c. evening,PHH,INACTIVE,150,1
1,20240805,20240805,Bangalore,PostOrderScreen,onTheWay,PostOrderOnTheWay1,a. morning,PHH,PRIME,21,0
2,20240714,20240708,Hyderabad,PostOrderScreen,arrived,PostOrderArrived1,a. morning,PHH,PLATINUM,1013,0
3,20240618,20240617,Jaipur,PostOrderScreen,onTheWay,PostOrderOnTheWay1,c. evening,PHH,GOLD,1171,3
4,20240712,20240708,Hyderabad,PostOrderScreen,started,PostOrderStarted1,c. evening,HH,HH,78,0


In [29]:
df_data.columns

Index(['yyyymmdd', 'week_date', 'city', 'pagename', 'screen_name', 'slotname',
       'time_temporal', 'ltr_segment', 'retention_segment', 'view', 'click'],
      dtype='object')

In [35]:
df_data.to_clipboard(index=False)

In [31]:
df_data.shape

(148623, 11)

### Analysis

In [43]:
df_analysis1 = df_data\
.groupby(['week_date'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis1) 

,week_date,view,click,ctr
0,20240617,11235237,20310,0.18
1,20240624,6765875,6746,0.10
2,20240701,8471882,13704,0.16
3,20240708,8411218,18307,0.22
4,20240715,9392339,18558,0.20
5,20240722,7055298,17591,0.25
6,20240729,1953984,6331,0.32
7,20240805,275360,2564,0.93
8,20240812,2812279,16127,0.57


In [58]:
df_analysis2 = df_data\
.groupby(['week_date', 'city'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis2) 
df_analysis2.pivot(index ='week_date' , columns ='city', values =['ctr'])

ctr                                              
city      Bangalore Chennai Delhi Hyderabad Jaipur Kolkata Mumbai
week_date                                                        
20240617       0.28    0.15  0.11      0.15   0.27    0.19   0.20
20240624       0.15    0.06  0.08      0.07   0.24    0.09   0.07
20240701       0.17    0.15  0.15      0.13   0.33    0.18   0.20
20240708       0.26    0.15  0.24      0.17   0.39    0.19   0.33
20240715       0.22    0.16  0.20      0.18   0.27    0.20   0.19
20240722       0.21    0.19  0.27      0.28   0.31    0.30   0.23
20240729       0.33    0.26  0.32      0.32   0.31    0.54   0.45
20240805       1.01    0.73  1.02      0.66   1.94    0.98   0.99
20240812       0.79    0.33  0.51      0.40   1.43    0.79   0.45

In [60]:
df_analysis3 = df_data\
.groupby(['pagename'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis3)

,pagename,view,click,ctr
0,CaptainSearchScreen,7730097,41103,0.53
1,HomeScreen,779780,1151,0.15
2,PostOrderScreen,47863595,77984,0.16


In [63]:
df_analysis4 = df_data\
.groupby(['pagename','screen_name'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis4)

,pagename,screen_name,view,click,ctr
0,CaptainSearchScreen,CaptainSearchScreen,7730097,41103,0.53
1,HomeScreen,HomeScreen,779780,1151,0.15
2,PostOrderScreen,arrived,11344951,9590,0.08
3,PostOrderScreen,onTheWay,24635041,40464,0.16
4,PostOrderScreen,started,11883603,27930,0.24


In [64]:
df_analysis5 = df_data\
.groupby(['time_temporal','screen_name'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis5) 
df_analysis5.pivot(index ='time_temporal' , columns ='screen_name', values =['ctr'])

ctr                                    
screen_name   CaptainSearchScreen HomeScreen arrived onTheWay started
time_temporal                                                        
a. morning                   0.49       0.14    0.07     0.14    0.22
b. afternoon                 0.47       0.17    0.08     0.14    0.24
c. evening                   0.64       0.14    0.08     0.14    0.23
d. rest                      0.42       0.17    0.14     0.32    0.28

In [66]:
df_analysis6 = df_data\
.groupby(['screen_name','slotname'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis6) 
df_analysis6.pivot(index = ['slotname'] , columns ='screen_name', values =['ctr'])

ctr                              \
screen_name             CaptainSearchScreen HomeScreen arrived onTheWay   
slotname                                                                  
CaptainSearch1                         0.53        NaN     NaN      NaN   
CaptainSearchScreenSlot                0.94        NaN     NaN      NaN   
HomeScreen1                             NaN       0.18     NaN      NaN   
HomeScreen2                             NaN       0.08     NaN      NaN   
HomeScreen3                             NaN       0.15     NaN      NaN   
HomeScreen4                             NaN       0.00     NaN      NaN   
PostOrderArrived1                       NaN        NaN    0.10      NaN   
PostOrderArrived2                       NaN        NaN    0.07      NaN   
PostOrderArrived3                       NaN        NaN    0.07      NaN   
PostOrderOnTheWay1                      NaN        NaN     NaN     0.24   
PostOrderOnTheWay2                      NaN        NaN     NaN     0.13   
PostOrderOnTheWay3                      NaN        NaN     NaN     0.10   
PostOrderScreenArrived                  NaN        NaN    0.24      NaN   
PostOrderScreenOnTheWay                 NaN        NaN     NaN     0.20   
PostOrderScreenStarted                  NaN        NaN     NaN      NaN   
PostOrderStarted1                       NaN        NaN     NaN      NaN   
PostOrderStarted2                       NaN        NaN     NaN      NaN   
PostOrderStarted3                       NaN        NaN     NaN      NaN   
PostOrderStarted4                       NaN        NaN     NaN      NaN   

                                 
screen_name             started  
slotname                         
CaptainSearch1              NaN  
CaptainSearchScreenSlot     NaN  
HomeScreen1                 NaN  
HomeScreen2                 NaN  
HomeScreen3                 NaN  
HomeScreen4                 NaN  
PostOrderArrived1           NaN  
PostOrderArrived2           NaN  
PostOrderArrived3           NaN  
PostOrderOnTheWay1          NaN  
PostOrderOnTheWay2          NaN  
PostOrderOnTheWay3          NaN  
PostOrderScreenArrived      NaN  
PostOrderScreenOnTheWay     NaN  
PostOrderScreenStarted     0.21  
PostOrderStarted1          0.28  
PostOrderStarted2          0.30  
PostOrderStarted3          0.07  
PostOrderStarted4          0.53

In [68]:
df_analysis7 = df_data\
.groupby(['screen_name','ltr_segment'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis7) 
df_analysis7.pivot(index = ['ltr_segment'] , columns ='screen_name', values =['ctr'])

ctr                                    
screen_name CaptainSearchScreen HomeScreen arrived onTheWay started
ltr_segment                                                        
HH                         0.82       0.22    0.14     0.23    0.37
PHH                        0.50       0.12    0.08     0.16    0.22
UNKNOWN                    1.67       0.35    0.40     0.39    0.42

In [70]:
df_analysis8 = df_data\
.groupby(['ltr_segment'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis8)

,ltr_segment,view,click,ctr
0,HH,4763247,15094,0.32
1,PHH,51242192,102754,0.20
2,UNKNOWN,368033,2390,0.65


In [72]:
df_analysis9 = df_data\
.groupby(['retention_segment'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis9)

,retention_segment,view,click,ctr
0,DORMANT,15262883,32595,0.21
1,ELITE,16063368,27397,0.17
2,GOLD,7846307,18392,0.23
3,HH,1087210,4647,0.43
4,INACTIVE,488968,2649,0.54
5,PLATINUM,12261132,24156,0.20
6,PRIME,678791,1231,0.18
7,SILVER,2316780,6781,0.29
8,UNKNOWN,368033,2390,0.65


In [74]:
df_analysis10 = df_data\
.groupby(['screen_name','retention_segment'])\
.agg(agg_dict)\
.reset_index()

calculate_percentage_columns(df_analysis10) 
df_analysis10.pivot(index = ['retention_segment'] , columns ='screen_name', values =['ctr'])

ctr                                    
screen_name       CaptainSearchScreen HomeScreen arrived onTheWay started
retention_segment                                                        
DORMANT                          0.57       0.16    0.10     0.18    0.27
ELITE                            0.42       0.09    0.07     0.13    0.18
GOLD                             0.55       0.14    0.09     0.18    0.26
HH                               0.78       0.19    0.15     0.26    0.41
INACTIVE                         1.26       0.24    0.39     0.32    0.24
PLATINUM                         0.48       0.11    0.07     0.15    0.21
PRIME                            0.40       0.08    0.06     0.14    0.22
SILVER                           0.61       0.15    0.11     0.21    0.28
UNKNOWN                          1.67       0.35    0.40     0.39    0.42